# ISIC 2019 Bronze ingestion

This notebook expects the ISIC 2019 Bronze tables/folders to already exist and the source archives to already be uploaded to landing.

Run `05_create_bronze_tables_and_folders.ipynb` before this notebook.


In [ ]:
%run ../_setup_env


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import BinaryType, LongType, StringType, StructField, StructType

from data_platform.files import extract_zip_members, iter_image_blob_rows, stage_archive_locally
from data_platform.setup import build_layout, load_dataset_config, load_yaml_config

DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "datasets" / "isic_2019.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
landing_paths = layout["landing_paths"]
bronze_paths = layout["bronze_paths"]
bronze_tables = layout["bronze_tables"]

ARCHIVES = [archive.copy() for archive in DATASET_CONFIG["archives"]]
LOCAL_ARCHIVE_STAGE_ROOT = Path("/local_disk0/tmp/isic_2019/archive_stage")
OVERWRITE_LOCAL_STAGE = False

for archive in ARCHIVES:
    source_split = archive["source_split"]
    archive["archive_dbfs_path"] = f"{landing_paths['dataset_archive'].rstrip('/')}/{archive['archive_filename']}"
    archive["archive_local_path"] = Path("/dbfs") / archive["archive_dbfs_path"].removeprefix("dbfs:/")
    archive["metadata_dir_path"] = f"{bronze_paths['metadata'].rstrip('/')}/{source_split}"
    archive["metadata_local_root"] = Path("/dbfs") / archive["metadata_dir_path"].removeprefix("dbfs:/")
    archive["metadata_target_path"] = f"{archive['metadata_dir_path'].rstrip('/')}/{archive['metadata_filename']}"

for archive in ARCHIVES:
    print(f"{archive['source_split']} archive landing path: {archive['archive_dbfs_path']}")
    print(f"{archive['source_split']} Bronze image rows table: {bronze_tables['image_blobs']}")
    print(f"{archive['source_split']} Bronze metadata path: {archive['metadata_target_path']}")


## Preflight

Verify that the required train and test archives already exist in landing before doing any extraction or table writes.


In [ ]:
for archive in ARCHIVES:
    if not archive["archive_local_path"].exists():
        raise FileNotFoundError(f"Missing archive: {archive['archive_dbfs_path']}")

print("All required source archives are present.")


## Stage Archives and Metadata

This step copies each source archive once from the landing Volume to local disk, then extracts only the embedded metadata CSV into Bronze metadata. Image extraction and byte reads happen from local disk in the next step.

In [ ]:
for archive in ARCHIVES:
    archive["staged_archive_path"] = stage_archive_locally(
        archive_path=archive["archive_local_path"],
        local_root=LOCAL_ARCHIVE_STAGE_ROOT,
        overwrite=OVERWRITE_LOCAL_STAGE,
    )
    extracted_metadata_files = extract_zip_members(
        archive_path=archive["staged_archive_path"],
        destination_root=archive["metadata_local_root"],
        predicate=lambda path, metadata_filename=archive["metadata_filename"]: path.name == metadata_filename,
    )
    if len(extracted_metadata_files) != 1:
        raise RuntimeError(
            f"Expected one {archive['metadata_filename']} in {archive['archive_filename']}, "
            f"extracted {len(extracted_metadata_files)}"
        )
    print(f"Staged archive locally: {archive['staged_archive_path']}")
    print(f"Copied metadata file into Bronze metadata: {archive['metadata_target_path']}")

display(dbutils.fs.ls(bronze_paths["metadata"]))


## Build And Write Image Blob Table

This step reads image bytes and checksums from the locally staged archives, then writes all image rows to one Delta table. It does not create one object per JPEG in the Volume.

In [ ]:
IMAGE_BLOB_SCHEMA = StructType(
    [
        StructField("image_id", StringType(), False),
        StructField("source_split", StringType(), False),
        StructField("archive_member_path", StringType(), False),
        StructField("source_archive_uri", StringType(), True),
        StructField("image_bytes", BinaryType(), False),
        StructField("byte_length", LongType(), False),
        StructField("source_checksum", StringType(), False),
    ]
)
IMAGE_BLOB_WRITE_BATCH_MAX_BYTES = 1024 * 1024 * 1024
IMAGE_BLOB_WRITE_BATCH_MAX_ROWS = 10_000

target_table = bronze_tables["image_blobs"]
seen_keys = set()  # (image_id, source_split) — dedup in memory, before any bytes hit storage
total_images_written = 0
first_batch = True

for archive in ARCHIVES:
    batch = []
    batch_bytes = 0
    archive_images_written = 0

    def flush(batch_rows, is_first):
        writer = spark.createDataFrame(batch_rows, schema=IMAGE_BLOB_SCHEMA).write.mode("overwrite" if is_first else "append")

        if is_first:
            writer = writer.option("overwriteSchema", "true")
        writer.saveAsTable(target_table)

    for row in iter_image_blob_rows(
        archive_path=archive["staged_archive_path"],
        source_split=archive["source_split"],
        source_archive_uri=archive["archive_dbfs_path"],
    ):
        key = (row["image_id"], row["source_split"])
        if key in seen_keys:
            continue
        seen_keys.add(key)

        batch.append(row)
        batch_bytes += row["byte_length"]
        if batch_bytes >= IMAGE_BLOB_WRITE_BATCH_MAX_BYTES or len(batch) >= IMAGE_BLOB_WRITE_BATCH_MAX_ROWS:
            flush(batch, first_batch)
            total_images_written += len(batch)
            archive_images_written += len(batch)
            first_batch = False
            batch = []
            batch_bytes = 0

    if batch:
        flush(batch, first_batch)
        total_images_written += len(batch)
        archive_images_written += len(batch)
        first_batch = False

    print(f"Wrote {archive_images_written} image rows for {archive['source_split']}")

image_blob_df = spark.table(target_table)
image_manifest_df = image_blob_df.select(
    F.col("image_id"),
    F.col("source_split"),
    F.concat(
        F.lit(f"table:{target_table}/"),
        F.col("source_split"),
        F.lit("/"),
        F.col("image_id"),
    ).alias("source_uri"),
    F.col("source_checksum"),
)

print(f"Images written to blob table: {total_images_written}")
display(image_blob_df.groupBy("source_split").count())


## Write Bronze Tables

This step loads the raw metadata file, aligns it with the image blob rows, merges the source rows into the Bronze table, and appends an ingestion run record.

In [ ]:
metadata_frames = [
    spark.read.option("header", True)
    .csv(archive["metadata_target_path"])
    .withColumn("source_split", F.lit(archive["source_split"]))
    for archive in ARCHIVES
]
raw_metadata_df = metadata_frames[0]
for metadata_frame in metadata_frames[1:]:
    raw_metadata_df = raw_metadata_df.unionByName(metadata_frame, allowMissingColumns=True)

source_metadata_df = (
    raw_metadata_df
    .withColumn("image_id", F.col("isic_id").cast("string"))
    .join(image_manifest_df, on=["image_id", "source_split"], how="left")
    .select(
        F.col("image_id"),
        F.col("source_split"),
        F.col("source_uri"),
        F.col("attribution"),
        F.col("copyright_license"),
        F.col("age_approx"),
        F.col("anatom_site_1"),
        F.col("anatom_site_2"),
        F.col("anatom_site_3"),
        F.col("anatom_site_4"),
        F.col("anatom_site_5"),
        F.col("anatom_site_special"),
        F.col("clin_size_long_diam_mm"),
        F.col("concomitant_biopsy"),
        F.col("dermoscopic_type"),
        F.col("diagnosis_1"),
        F.col("diagnosis_2"),
        F.col("diagnosis_3"),
        F.col("diagnosis_4"),
        F.col("diagnosis_5"),
        F.col("diagnosis_confirm_type"),
        F.col("family_hx_mm"),
        F.col("image_type"),
        F.col("lesion_id"),
        F.col("melanocytic"),
        F.col("patient_id"),
        F.col("personal_hx_mm"),
        F.col("sex"),
        F.col("source_checksum"),
    )
    .where(F.col("image_id").isNotNull())
)

ingestion_run_id = f"bronze-{uuid4().hex[:12]}"
started_at = datetime.now(timezone.utc)
source_metadata_df = (
    source_metadata_df
    .withColumn("ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("ingested_at", F.lit(started_at))
)

source_metadata_df.createOrReplaceTempView("staged_isic_2019_source_metadata")
spark.sql(
    f"""
    MERGE INTO {bronze_tables['source_metadata']} AS target
    USING staged_isic_2019_source_metadata AS source
    ON target.image_id = source.image_id AND target.source_split = source.source_split
    WHEN MATCHED THEN UPDATE SET
      target.source_split = source.source_split,
      target.source_uri = source.source_uri,
      target.attribution = source.attribution,
      target.copyright_license = source.copyright_license,
      target.age_approx = source.age_approx,
      target.anatom_site_1 = source.anatom_site_1,
      target.anatom_site_2 = source.anatom_site_2,
      target.anatom_site_3 = source.anatom_site_3,
      target.anatom_site_4 = source.anatom_site_4,
      target.anatom_site_5 = source.anatom_site_5,
      target.anatom_site_special = source.anatom_site_special,
      target.clin_size_long_diam_mm = source.clin_size_long_diam_mm,
      target.concomitant_biopsy = source.concomitant_biopsy,
      target.dermoscopic_type = source.dermoscopic_type,
      target.diagnosis_1 = source.diagnosis_1,
      target.diagnosis_2 = source.diagnosis_2,
      target.diagnosis_3 = source.diagnosis_3,
      target.diagnosis_4 = source.diagnosis_4,
      target.diagnosis_5 = source.diagnosis_5,
      target.diagnosis_confirm_type = source.diagnosis_confirm_type,
      target.family_hx_mm = source.family_hx_mm,
      target.image_type = source.image_type,
      target.lesion_id = source.lesion_id,
      target.melanocytic = source.melanocytic,
      target.patient_id = source.patient_id,
      target.personal_hx_mm = source.personal_hx_mm,
      target.sex = source.sex,
      target.source_checksum = source.source_checksum,
      target.ingestion_run_id = source.ingestion_run_id,
      target.ingested_at = source.ingested_at
    WHEN NOT MATCHED THEN INSERT *
    """
)

records_written = source_metadata_df.count()
finished_at = datetime.now(timezone.utc)
run_row = [{
    "ingestion_run_id": ingestion_run_id,
    "dataset_name": DATASET_CONFIG["dataset_name"],
    "source_version": DATASET_CONFIG["source_version"],
    "started_at": started_at,
    "finished_at": finished_at,
    "status": "success",
    "records_seen": records_written,
    "records_written": records_written,
}]
spark.createDataFrame(run_row).write.mode("append").saveAsTable(bronze_tables["ingestion_runs"])

print(f"Ingestion run id: {ingestion_run_id}")
print(f"Records written: {records_written}")


## Review Outputs

This final step shows the landed archive, Bronze file paths, and the most recent Bronze table contents for inspection.

In [ ]:
print("Landing archive contents:")
display(dbutils.fs.ls(landing_paths["dataset_archive"]))

print("Bronze metadata path contents:")
display(dbutils.fs.ls(bronze_paths["metadata"]))

print("Bronze image blob rows by split:")
display(spark.table(bronze_tables["image_blobs"]).groupBy("source_split").count())

print("Recent ingestion runs:")
display(spark.table(bronze_tables["ingestion_runs"]).orderBy("started_at", ascending=False))

print("Source metadata sample:")
display(spark.table(bronze_tables["source_metadata"]))
